# 🧊 Analyse des Offres d'Emploi LinkedIn
**ESME — Architecture Big Data 2026**  
**Binôme** : Joshua Maarek & Harold Malherbe

Ce notebook présente toutes les étapes du projet : création de la base, chargement des données depuis S3, et analyses du marché de l'emploi LinkedIn.

In [ ]:
-- Étape 1 : Création de la base de données
CREATE DATABASE IF NOT EXISTS linkedin;
USE DATABASE linkedin;
USE SCHEMA public;

## 🔗 Étape 2 — Création du Stage S3 et des formats de fichiers

On crée un **stage externe** pointant vers le bucket S3 public `snowflake-lab-bucket`. 
Le stage permet à Snowflake d'accéder aux fichiers distants sans les télécharger localement.
On définit également deux formats : `csv_format` pour les CSV et `json_format` pour les JSON.

In [ ]:
CREATE OR REPLACE STAGE linkedin_stage
  URL = 's3://snowflake-lab-bucket/'
  FILE_FORMAT = (TYPE = 'CSV');

CREATE OR REPLACE FILE FORMAT csv_format
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1
  NULL_IF = ('', 'NULL', 'null')
  EMPTY_FIELD_AS_NULL = TRUE
  TRIM_SPACE = TRUE;

CREATE OR REPLACE FILE FORMAT json_format
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE;

## 📋 Étape 3 — Création des tables

On crée les 8 tables correspondant aux fichiers du dataset. 
Ajustements effectués par rapport au schéma initial :
- `zip_code` en `VARCHAR(200)` — certaines valeurs contiennent du texte long
- `speciality` en `TEXT` — certaines valeurs dépassent 1000 caractères

In [ ]:
CREATE OR REPLACE TABLE job_postings (
    job_id                    VARCHAR(50),
    company_name              VARCHAR(255),
    title                     VARCHAR(500),
    description               TEXT,
    max_salary                FLOAT,
    med_salary                FLOAT,
    min_salary                FLOAT,
    pay_period                VARCHAR(50),
    formatted_work_type       VARCHAR(100),
    location                  VARCHAR(255),
    applies                   INTEGER,
    original_listed_time      BIGINT,
    remote_allowed            BOOLEAN,
    views                     INTEGER,
    job_posting_url           VARCHAR(1000),
    application_url           VARCHAR(1000),
    application_type          VARCHAR(100),
    expiry                    BIGINT,
    closed_time               BIGINT,
    formatted_experience_level VARCHAR(100),
    skills_desc               TEXT,
    listed_time               BIGINT,
    posting_domain            VARCHAR(255),
    sponsored                 BOOLEAN,
    work_type                 VARCHAR(100),
    currency                  VARCHAR(10),
    compensation_type         VARCHAR(100)
);

CREATE OR REPLACE TABLE benefits (
    job_id      VARCHAR(50),
    inferred    BOOLEAN,
    type        VARCHAR(200)
);

CREATE OR REPLACE TABLE companies (
    company_id    VARCHAR(50),
    name          VARCHAR(500),
    description   TEXT,
    company_size  INTEGER,
    state         VARCHAR(100),
    country       VARCHAR(100),
    city          VARCHAR(200),
    zip_code      VARCHAR(200),
    address       VARCHAR(500),
    url           VARCHAR(1000)
);

CREATE OR REPLACE TABLE employee_counts (
    company_id      VARCHAR(50),
    employee_count  INTEGER,
    follower_count  INTEGER,
    time_recorded   BIGINT
);

CREATE OR REPLACE TABLE job_skills (
    job_id      VARCHAR(50),
    skill_abr   VARCHAR(50)
);

CREATE OR REPLACE TABLE job_industries (
    job_id       VARCHAR(50),
    industry_id  VARCHAR(50)
);

CREATE OR REPLACE TABLE company_specialities (
    company_id  VARCHAR(50),
    speciality  TEXT
);

CREATE OR REPLACE TABLE company_industries (
    company_id  VARCHAR(50),
    industry    VARCHAR(1000)
);

## 📥 Étape 4 — Chargement des données

On utilise `COPY INTO` pour charger les fichiers CSV directement depuis le stage S3.
Pour les fichiers JSON, on passe par une table temporaire `VARIANT` pour parser les champs.

In [ ]:
COPY INTO job_postings
FROM @linkedin_stage/job_postings.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR = 'CONTINUE';

COPY INTO benefits
FROM @linkedin_stage/benefits.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR = 'CONTINUE';

COPY INTO employee_counts
FROM @linkedin_stage/employee_counts.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR = 'CONTINUE';

COPY INTO job_skills
FROM @linkedin_stage/job_skills.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR = 'CONTINUE';

CREATE OR REPLACE TEMP TABLE companies_raw (v VARIANT);
COPY INTO companies_raw
FROM @linkedin_stage/companies.json
FILE_FORMAT = (FORMAT_NAME = 'json_format');

INSERT INTO companies
SELECT
    v:company_id::VARCHAR, v:name::VARCHAR, v:description::VARCHAR,
    v:company_size::INTEGER, v:state::VARCHAR, v:country::VARCHAR,
    v:city::VARCHAR, v:zip_code::VARCHAR(200), v:address::VARCHAR, v:url::VARCHAR
FROM companies_raw;

CREATE OR REPLACE TEMP TABLE company_industries_raw (v VARIANT);
COPY INTO company_industries_raw
FROM @linkedin_stage/company_industries.json
FILE_FORMAT = (FORMAT_NAME = 'json_format');

INSERT INTO company_industries
SELECT v:company_id::VARCHAR, v:industry::VARCHAR
FROM company_industries_raw;

CREATE OR REPLACE TEMP TABLE company_specialities_raw (v VARIANT);
COPY INTO company_specialities_raw
FROM @linkedin_stage/company_specialities.json
FILE_FORMAT = (FORMAT_NAME = 'json_format');

ALTER TABLE company_specialities MODIFY COLUMN speciality TEXT;
INSERT INTO company_specialities
SELECT v:company_id::VARCHAR, v:speciality::VARCHAR
FROM company_specialities_raw;

CREATE OR REPLACE TEMP TABLE job_industries_raw (v VARIANT);
COPY INTO job_industries_raw
FROM @linkedin_stage/job_industries.json
FILE_FORMAT = (FORMAT_NAME = 'json_format');

INSERT INTO job_industries
SELECT v:job_id::VARCHAR, v:industry_id::VARCHAR
FROM job_industries_raw;

## 📊 Analyse 1 — Top 10 des titres les plus publiés par industrie

Jointure entre `job_postings` et `job_industries`. On utilise `QUALIFY ROW_NUMBER()` pour garder uniquement le top 10 des titres les plus fréquents par industrie.

In [ ]:
SELECT ji.industry_id AS INDUSTRY, jp.title AS JOB_TITLE, COUNT(*) AS NB_POSTINGS
FROM job_postings jp
JOIN job_industries ji ON jp.job_id = ji.job_id
WHERE jp.title IS NOT NULL
GROUP BY ji.industry_id, jp.title
QUALIFY ROW_NUMBER() OVER (PARTITION BY ji.industry_id ORDER BY COUNT(*) DESC) <= 10
ORDER BY INDUSTRY, NB_POSTINGS DESC;

## 📊 Analyse 2 — Top 10 des postes les mieux rémunérés par industrie

Même jointure que l'analyse 1. On filtre sur `pay_period = 'YEARLY'` pour comparer des salaires annuels cohérents, et on calcule la moyenne de `max_salary` par titre et industrie.

In [ ]:
SELECT ji.industry_id AS INDUSTRY, jp.title AS JOB_TITLE,
    ROUND(AVG(jp.max_salary),0) AS AVG_MAX_SALARY
FROM job_postings jp
JOIN job_industries ji ON jp.job_id = ji.job_id
WHERE jp.max_salary IS NOT NULL AND jp.pay_period = 'YEARLY'
GROUP BY ji.industry_id, jp.title
QUALIFY ROW_NUMBER() OVER (PARTITION BY ji.industry_id ORDER BY AVG(jp.max_salary) DESC) <= 10
ORDER BY INDUSTRY, AVG_MAX_SALARY DESC;

## 📊 Analyse 3 — Répartition des offres par taille d'entreprise

Jointure `job_postings` ↔ `companies` via `company_name`. Le champ `company_size` (0 à 7) est mappé vers des libellés lisibles. Résultat : la majorité des offres proviennent d'entreprises dont la taille n'est pas renseignée.

In [ ]:
SELECT
    CASE c.company_size
        WHEN 0 THEN '0 - Tres petite'
        WHEN 1 THEN '1 - Petite'
        WHEN 2 THEN '2 - Petite-moyenne'
        WHEN 3 THEN '3 - Moyenne'
        WHEN 4 THEN '4 - Moyenne-grande'
        WHEN 5 THEN '5 - Grande'
        WHEN 6 THEN '6 - Tres grande'
        WHEN 7 THEN '7 - Entreprise'
        ELSE 'Non renseigne'
    END AS COMPANY_SIZE_LABEL,
    COUNT(jp.job_id) AS NB_OFFRES
FROM job_postings jp
LEFT JOIN companies c ON jp.company_name = c.name
GROUP BY c.company_size, COMPANY_SIZE_LABEL
ORDER BY NB_OFFRES DESC;

## 📊 Analyse 4 — Répartition des offres par secteur d'activité

Jointure `job_postings` ↔ `job_industries`. On compte les offres distinctes par `industry_id` et on affiche les 20 premiers secteurs par volume.

In [ ]:
SELECT ji.industry_id AS INDUSTRY, COUNT(DISTINCT jp.job_id) AS NB_OFFRES
FROM job_postings jp
JOIN job_industries ji ON jp.job_id = ji.job_id
GROUP BY ji.industry_id
ORDER BY NB_OFFRES DESC
LIMIT 20;

## 📊 Analyse 5 — Répartition des offres par type d'emploi

Utilisation directe de `formatted_work_type` dans `job_postings`. On calcule le pourcentage de chaque type via une fenêtre `OVER ()`.

Résultat : **82.57% Full-time**, 8.84% Contract, 6.81% Part-time.

In [ ]:
SELECT COALESCE(formatted_work_type, 'Non renseigne') AS WORK_TYPE,
    COUNT(*) AS NB_OFFRES,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS POURCENTAGE
FROM job_postings
GROUP BY formatted_work_type
ORDER BY NB_OFFRES DESC;